# 1.5b Reddit — Sentiment Analysis (Twitter-RoBERTa)

**Temporary exploratory notebook** — applies `cardiffnlp/twitter-roberta-base-sentiment-latest`
to Reddit posts + comments. Same graph structure as `2.5_sentiment_analysis.ipynb` (Bluesky).

| Field | Details |
|---|---|
| **Model** | `cardiffnlp/twitter-roberta-base-sentiment-latest` |
| **Labels** | Negative · Neutral · Positive |
| **Compound** | `roberta_sentiment` = `roberta_pos − roberta_neg` ∈ [−1, +1] |
| **Input** | `Data/2_Silver/Reddit/reddit_posts_clean.parquet` + `reddit_comments_clean.parquet` |
| **Output** | `Data/2_Silver/Reddit/sentiment_reddit_roberta.csv` |

> **Note:** For 1.4 M rows this takes ~30–60 min on CPU.
> Set `SAMPLE_N` below to e.g. `50_000` for a quick test run.

In [ ]:
import os, sys, warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import matplotlib.lines as mlines

sys.path.insert(0, os.path.abspath('../..'))
from house_style import (
    apply_style, REPUBLICAN, DEMOCRAT, NEUTRAL, ACCENT,
    TEXT_PRIMARY, TEXT_MUTED, BG_DARK, BG_PANEL,
    EVENTS, add_events,
)
apply_style()

# ── Paths ──────────────────────────────────────────────────────────────────────
POSTS_PATH    = '../../Data/2_Silver/Reddit/reddit_posts_clean.parquet'
COMMENTS_PATH = '../../Data/2_Silver/Reddit/reddit_comments_clean.parquet'
VADER_CSV     = '../../Data/2_Silver/Reddit/sentiment_reddit.csv'

# Set to None to process all rows; set to e.g. 50_000 for a quick test
SAMPLE_N   = None
BATCH_SIZE = 64
MODEL_NAME = 'cardiffnlp/twitter-roberta-base-sentiment-latest'

# ── Colour helpers ─────────────────────────────────────────────────────────────
BUZZ_COLORS = {'TrumpBuzz': REPUBLICAN, 'HarrisBuzz': DEMOCRAT, 'ElectionBuzz': NEUTRAL}
BUZZ_ORDER  = ['TrumpBuzz', 'HarrisBuzz', 'ElectionBuzz']
POL_COLORS  = {'Positive': '#2ecc71', 'Neutral': TEXT_MUTED, 'Negative': REPUBLICAN}
PURPLE      = '#9b5de5'

print('Setup complete.')

## 1. Load data

In [ ]:
posts    = pd.read_parquet(POSTS_PATH)
comments = pd.read_parquet(COMMENTS_PATH)

posts['source']    = 'post'
comments['source'] = 'comment'

keep = ['id', 'author', 'created_utc', 'subreddit', 'score',
        'text', 'text_clean', 'word_count', 'candidate', 'source']

df = pd.concat([posts[keep], comments[keep]], ignore_index=True)
df['date']       = pd.to_datetime(df['created_utc'], utc=True).dt.normalize()
df['text']       = df['text'].fillna('')
df['text_clean'] = df['text_clean'].fillna('')

df = df[(df['date'] >= '2024-07-05') & (df['date'] <= '2024-11-04')].reset_index(drop=True)

if SAMPLE_N is not None:
    df = df.sample(n=min(SAMPLE_N, len(df)), random_state=42).reset_index(drop=True)
    print(f'>>> SAMPLE mode: using {len(df):,} rows <<<')

print(f'Total rows : {len(df):,}  ({len(posts):,} posts + {len(comments):,} comments)')
print(f'Date range : {df["date"].min().date()} → {df["date"].max().date()}')
print()
print(df['candidate'].value_counts())

---
## 2. Twitter-RoBERTa

**What it is:** `cardiffnlp/twitter-roberta-base-sentiment-latest` is a RoBERTa transformer
fine-tuned on ~124 million tweets. Unlike VADER and TextBlob (which match words against a fixed
lexicon), RoBERTa reads the full sentence in context — it understands that *"not great"* is
negative, even though *"great"* alone is positive.

**Output:** Probabilities for Negative / Neutral / Positive. We take
`roberta_pos − roberta_neg` as a single sentiment score ∈ [−1, +1],
analogous to VADER's compound score.

In [ ]:
from transformers import pipeline

print('Loading Twitter-RoBERTa model (downloads once, then cached)...')
roberta = pipeline(
    'sentiment-analysis',
    model=MODEL_NAME,
    tokenizer=MODEL_NAME,
    max_length=512,
    truncation=True,
    device=-1,   # CPU; change to 0 if you have a GPU
)
print('Model loaded.')

In [ ]:
print('Running Twitter-RoBERTa on all posts (this takes a few minutes)...')

texts = df['text'].fillna('').tolist()
pos_probs, neg_probs, neu_probs, labels = [], [], [], []

for i in range(0, len(texts), BATCH_SIZE):
    batch   = [str(t)[:512] for t in texts[i : i + BATCH_SIZE]]
    results = roberta(batch, top_k=None)
    for r in results:
        probs = {x['label'].lower(): x['score'] for x in r}
        pos_probs.append(probs.get('positive', 0.0))
        neg_probs.append(probs.get('negative', 0.0))
        neu_probs.append(probs.get('neutral',  0.0))
        labels.append(max(probs, key=probs.get).capitalize())
    if (i // BATCH_SIZE) % 10 == 0:
        print(f'  {i}/{len(texts)} rows processed...')

df['roberta_pos']       = pos_probs
df['roberta_neg']       = neg_probs
df['roberta_neu']       = neu_probs
df['roberta_sentiment'] = df['roberta_pos'] - df['roberta_neg']
df['roberta_label']     = labels

print(f'\nDone. {len(df):,} rows scored.')
print(df['roberta_label'].value_counts())
print(f'Mean sentiment: {df["roberta_sentiment"].mean():.4f}')

### Distribution of sentiment scores

In [ ]:
fig, ax = plt.subplots(figsize=(10, 4))
fig.patch.set_facecolor(BG_DARK)
ax.set_facecolor(BG_PANEL)
ax.hist(df['roberta_sentiment'], bins=40, color=DEMOCRAT, alpha=0.85, edgecolor='none')
ax.axvline(0, color='white', linestyle='--', linewidth=1, alpha=0.6, label='Zero')
ax.axvline(df['roberta_sentiment'].mean(), color='yellow', linewidth=1.5,
           label=f'Mean: {df["roberta_sentiment"].mean():.4f}')
ax.set_title('Twitter-RoBERTa — Distribution of Sentiment Scores',
             color=TEXT_PRIMARY, fontsize=12, fontweight='bold')
ax.set_xlabel('RoBERTa sentiment (pos − neg probability)', color=TEXT_MUTED)
ax.set_ylabel('Count', color=TEXT_MUTED)
ax.tick_params(colors=TEXT_MUTED)
ax.legend(facecolor=BG_DARK, labelcolor=TEXT_PRIMARY)
plt.tight_layout(); plt.show()

### Polarity share per buzz group

In [ ]:
label_shares = (
    df.groupby(['candidate', 'roberta_label']).size()
      .unstack(fill_value=0)
      .apply(lambda r: r / r.sum(), axis=1)
      .reindex(BUZZ_ORDER)
)
for col in ['Positive', 'Neutral', 'Negative']:
    if col not in label_shares.columns:
        label_shares[col] = 0.0
label_shares = label_shares[['Positive', 'Neutral', 'Negative']]

fig, ax = plt.subplots(figsize=(8, 4))
fig.patch.set_facecolor(BG_DARK); ax.set_facecolor(BG_PANEL)
x, bottom = np.arange(len(BUZZ_ORDER)), np.zeros(len(BUZZ_ORDER))
for label, color in POL_COLORS.items():
    vals = label_shares[label].values
    ax.bar(x, vals, bottom=bottom, color=color, label=label, edgecolor='none')
    for i, (v, b) in enumerate(zip(vals, bottom)):
        if v > 0.05:
            ax.text(x[i], b + v/2, f'{v:.0%}', ha='center', va='center',
                    color='white', fontsize=9)
    bottom += vals
ax.set_xticks(x); ax.set_xticklabels(BUZZ_ORDER, color=TEXT_MUTED)
ax.set_ylabel('Share', color=TEXT_MUTED)
ax.set_title('Twitter-RoBERTa — Polarity Share per Buzz Group',
             color=TEXT_PRIMARY, fontsize=12, fontweight='bold')
ax.tick_params(colors=TEXT_MUTED)
ax.legend(facecolor=BG_DARK, labelcolor=TEXT_PRIMARY, loc='upper right')
plt.tight_layout(); plt.show()

### Daily sentiment per buzz group (7d rolling avg)

In [ ]:
fig, ax = plt.subplots(figsize=(13, 4))
fig.patch.set_facecolor(BG_DARK); ax.set_facecolor(BG_PANEL)
for group, color in BUZZ_COLORS.items():
    daily = df[df['candidate'] == group].groupby('date')['roberta_sentiment'].mean()
    roll  = daily.rolling(7, min_periods=1).mean()
    ax.plot(daily.index, daily.values, color=color, alpha=0.15, linewidth=1)
    ax.plot(roll.index, roll.values, color=color, linewidth=2, label=group)
ax.axhline(0, color='white', linestyle='--', linewidth=0.8, alpha=0.5)
add_events(ax)
ax.set_title('Twitter-RoBERTa — Daily Sentiment per Buzz Group (7d rolling avg)',
             color=TEXT_PRIMARY, fontsize=12, fontweight='bold')
ax.set_ylabel('RoBERTa sentiment (pos − neg)', color=TEXT_MUTED)
ax.xaxis.set_major_formatter(mdates.DateFormatter('%b %Y'))
ax.tick_params(axis='x', rotation=30, colors=TEXT_MUTED)
ax.tick_params(axis='y', colors=TEXT_MUTED)
data_handles = ax.get_legend_handles_labels()
event_handles = [
    mlines.Line2D([], [], color=c, linestyle=':', linewidth=1.8, label=lbl)
    for lbl, _, c in EVENTS
]
ax.legend(handles=data_handles[0], labels=data_handles[1],
          facecolor=BG_DARK, labelcolor=TEXT_PRIMARY, loc='upper left')
fig.legend(
    handles=event_handles, loc='lower center', bbox_to_anchor=(0.5, 0.0),
    ncol=min(len(EVENTS), 4), facecolor=BG_PANEL, edgecolor=TEXT_MUTED,
    labelcolor=TEXT_PRIMARY, fontsize=8.5, framealpha=0.95, borderpad=0.8,
)
plt.tight_layout(rect=[0, 0.18, 1, 1]); plt.show()

---
## 3. Comparison with VADER (z-scored overlay)

How well do RoBERTa and VADER agree? Z-scored daily sentiment overlay + correlation.

In [ ]:
vader_df      = pd.read_csv(VADER_CSV, usecols=['date', 'vader_compound'], parse_dates=['date'])
vader_daily   = vader_df.groupby('date')['vader_compound'].mean()
roberta_daily = df.groupby('date')['roberta_sentiment'].mean()

sent_cols = {'RoBERTa': roberta_daily, 'VADER': vader_daily}
METHOD_COLORS = {'RoBERTa': DEMOCRAT, 'VADER': ACCENT}

fig, ax = plt.subplots(figsize=(13, 5))
fig.patch.set_facecolor(BG_DARK); ax.set_facecolor(BG_PANEL)
for label, series in sent_cols.items():
    roll = series.rolling(7, min_periods=1).mean()
    z    = (roll - roll.mean()) / (roll.std() + 1e-8)
    ax.plot(z.index, z.values, linewidth=2, label=label, color=METHOD_COLORS[label])
ax.axhline(0, color='white', linestyle='--', linewidth=0.8, alpha=0.5)
add_events(ax)
ax.set_title('RoBERTa vs VADER — Daily Sentiment (7d rolling avg, z-scored)',
             color=TEXT_PRIMARY, fontsize=12, fontweight='bold')
ax.set_ylabel('Z-scored sentiment', color=TEXT_MUTED)
ax.xaxis.set_major_formatter(mdates.DateFormatter('%b %Y'))
ax.tick_params(axis='x', rotation=30, colors=TEXT_MUTED)
ax.tick_params(axis='y', colors=TEXT_MUTED)
data_handles = ax.get_legend_handles_labels()
event_handles = [
    mlines.Line2D([], [], color=c, linestyle=':', linewidth=1.8, label=lbl)
    for lbl, _, c in EVENTS
]
ax.legend(handles=data_handles[0], labels=data_handles[1],
          facecolor=BG_DARK, labelcolor=TEXT_PRIMARY, loc='upper left')
fig.legend(
    handles=event_handles, loc='lower center', bbox_to_anchor=(0.5, 0.0),
    ncol=min(len(EVENTS), 4), facecolor=BG_PANEL, edgecolor=TEXT_MUTED,
    labelcolor=TEXT_PRIMARY, fontsize=8.5, framealpha=0.95, borderpad=0.8,
)
plt.tight_layout(rect=[0, 0.18, 1, 1]); plt.show()

merged = pd.concat([roberta_daily.rename('RoBERTa'), vader_daily.rename('VADER')], axis=1).dropna()
print('Correlation matrix:')
print(merged.corr().round(3))

---
## 4. Save (temporary)

In [ ]:
OUT = '../../Data/2_Silver/Reddit/sentiment_reddit_roberta.csv'
save_cols = ['date', 'candidate', 'subreddit', 'source',
             'roberta_sentiment', 'roberta_pos', 'roberta_neg', 'roberta_neu',
             'roberta_label']
df[save_cols].to_csv(OUT, index=False)
print(f'Saved {len(df):,} rows → {OUT}')